In [2]:
!pip install openai python-dotenv chromadb sentence-transformers


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import chromadb
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.PersistentClient(path = "D:/sudhendra/learning projects/GraphRAG/vector_db")

collection = chroma_client.get_collection(
    name = "apple23_10k"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10208.50it/s]


In [8]:
def retrieve_chunks(query,n_results = 5):
    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings = [query_embedding],
        n_results = n_results
    )

    retrieved = []

    for i in range(len(results["documents"][0])):
        retrieved.append({
            "text" : results["documents"][0][i],
            "page" : results["metadatas"][0][i]["page"],
            "distance" : results["distances"][0][i]
        })

    return retrieved

In [9]:
chunks = retrieve_chunks("What are Apple's business segments?", n_results=5)

for chunk in chunks:
    print("Page:", chunk["page"])
    print("Distance:", chunk["distance"])
    print(chunk["text"][:700])
    print("-" * 80)

Page: 11
Distance: 0.7530658841133118
products and services, delay in new product and service introductions and lost sales.
Apple Inc. | 2023 Form 10-K | 8
--------------------------------------------------------------------------------
Page: 4
Distance: 0.7715369462966919
particular years, quarters, months or periods refer to the Company’s fiscal years ended in September and the associated 
quarters, months and periods of those fiscal years. Each of the terms the “Company” and “Apple” as used herein refers 
collectively to Apple Inc. and its wholly owned subsidiaries, unless otherwise stated.
PART I
Item 1. 
Business
Company Background
The Company designs, manufactures and markets smartphones, personal computers, tablets, wearables and accessories, and 
sells a variety of related services. The Company’s fiscal year is the 52- or 53-week period that ends on the last Saturday of 
September.
Products
iPhone
iPhone® is the Company’s line of smartphones based on i
-------------------------

In [12]:
def build_context(chunks):
    context = ""

    for i,chunk in enumerate(chunks,start = 1):
        context += f"\n[Source{i} | Page {chunk['page']}]\n"
        context += chunk["text"]
        context += "\n"
    
    return context

In [13]:
!pip install ollama


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import ollama

def generate_answer(query, n_results=5):
    chunks = retrieve_chunks(query, n_results=n_results)
    context = build_context(chunks)

    prompt = f"""
You are a financial report assistant.

Answer the user's question using ONLY the provided Apple 2023 Form 10-K context.

Rules:
1. Do not use outside knowledge.
2. If the answer is not present in the context, say:
   "The provided report context does not contain enough information."
3. Give page references.
4. Keep the answer clear and financial-analysis oriented.

Context:
{context}

Question:
{query}

Answer:
Evidence:
Page reference:
Confidence:
"""

    response = ollama.chat(
        model="mistral",
        messages=[
            {"role": "user", "content": prompt}
        ],
        options={
            "temperature": 0
        }
    )

    return response["message"]["content"]

In [19]:
answer = generate_answer("What were Apple's net sales in 2023?", n_results=5)
print(answer)

 The provided report context states that Apple's total net sales for 2023 were $383.3 billion [Source4 | Page 51].

Reference(s): Source4, Page 51


In [20]:
pip install pdfplumber camelot-py pandas

  Using cached pillow-12.2.0-cp313-cp313-win_amd64.whl.metadata (9.0 kB)
  Using cached cffi-2.0.0-cp313-cp313-win_amd64.whl.metadata (2.6 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
   ---------------------------------------- 0.0/6.6 MB ? eta -:--:--
   --------------------------------- ------ 5.5/6.6 MB 27.7 MB/s eta 0:00:01
   ---------------------------------------- 6.6/6.6 MB 23.1 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   --------------------------- ------------ 2.6/3.8 MB 13.5 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 12.7 MB/s  0:00:00
Using cached cffi-2.0.0-cp313-cp313-win_amd64.whl (183 kB)
   ---------------------------------------- 0.0/40.1 MB ? eta -:--:--
   -- ------------------------------------- 2.4/40.1 MB 12.5 MB/s eta 0:00:04
   ----- ---------------------------------- 5.2/40.1 MB 13.5 MB/s eta 0:00:03
 


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import ollama

def generate_answer(query, n_results=5):
    chunks = retrieve_chunks(query, n_results=n_results)
    context = build_context(chunks)

    prompt = f"""
You are a financial report assistant.

Answer the user's question using ONLY the provided Apple 2023 Form 10-K context.

Rules:
1. Do not use outside knowledge.
2. Do not invent missing numbers.
3. Do not calculate a year value unless both required values are explicitly present in the context.
4. For comparison questions, first extract directly stated values from tables or text.
5. If exact values are missing, say the retrieved context is insufficient.
6. Show calculation steps only after confirming the numbers are present in context.

Context:
{context}

Question:
{query}

Answer in this format:
2023 value:
2022 value:
Difference:
Percentage change:
Source:
Limitations:
"""

    response = ollama.chat(
        model="mistral",
        messages=[
            {"role": "user", "content": prompt}
        ],
        options={
            "temperature": 0
        }
    )

    return response["message"]["content"]

In [ ]:
answer = generate_answer("Apple total net sales 2023 2022 2021 table", n_results=10)
print(answer)
#Financial RAG needs table-aware retrieval.

 2023 value: $383,285
2022 value: $394,328
Difference: -$11,043
Percentage change: -2.8%
Source: [Source3 | Page 23] and [Source5 | Page 25]
Limitations: The provided context does not explicitly state the total net sales for each year in a table format. Instead, it is calculated by summing up the net sales of individual products and services from another source ([Source5 | Page 25]). This might lead to potential rounding errors when comparing the values directly. However, the percentage change calculation should still be accurate as it uses the same rounded numbers.
